# PyTorch inference with Ray Data

***

## Prerequisites

In [ ]:
%pip install -r ./scripts/requirements.txt --upgrade

In [ ]:
# Copy Ray launcher script to the scripts directory. 
%cp ../../../scripts/launcher.py ./scripts/

***

# Step 1 - Import Modules

Here we’ll import some libraries and define some variables.

In [ ]:
import os

# os.environ["AWS_PROFILE"] = "<aws_profile>"

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import get_execution_role, Session

In [ ]:
sagemaker_client = boto3.client("sagemaker")
s3_client = boto3.client("s3")

Create a SageMaker Session and save the default region and the execution role in some Python variables

In [ ]:
sagemaker_session = Session()
region = sagemaker_session.boto_session.region_name
role = get_execution_role()

In [ ]:
bucket_name = sagemaker_session.default_bucket()

***

## (Optional) Copy Prometheus binary

In case you want to avoid Ray to download prometheus, you can copy the binary on S3 and pass as parameter to the Training job

In [ ]:
! wget https://github.com/prometheus/prometheus/releases/download/v3.13.1/prometheus-3.13.1.linux-amd64.tar.gz

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session

In [ ]:
sagemaker_session = Session()
s3_client = boto3.client('s3')

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-modeltrainer-sft-ray"
else:
    input_path = f"datasets/llm-fine-tuning-modeltrainer-sft-ray"

prometheus_s3_path = (
    f"s3://{bucket_name}/{input_path}/prometheus/prometheus-3.13.1.linux-amd64.tar.gz"
)

In [ ]:
s3_client.upload_file(
    "./prometheus-3.13.1.linux-amd64.tar.gz",
    bucket_name,
    f"{input_path}/prometheus/prometheus-3.13.1.linux-amd64.tar.gz",
)

print(f"Prometheus binary uploaded to:")
print(prometheus_s3_path)

***

## (Optional) Copy Grafana binary

In case you want to render the metrics without an external Grafana, you can copy the Grafana binary on S3 and pass as parameter to the Training job. The launcher starts it on the head node and wires it into the **Metrics** tab of the Ray Dashboard.

This is normally combined with the Prometheus binary above: on a cluster without internet access Ray can download neither of them.

In [ ]:
! wget https://dl.grafana.com/oss/release/grafana-12.0.1.linux-amd64.tar.gz

In [ ]:
grafana_s3_path = (
    f"s3://{bucket_name}/{input_path}/grafana/grafana-12.0.1.linux-amd64.tar.gz"
)

In [ ]:
s3_client.upload_file(
    "./grafana-12.0.1.linux-amd64.tar.gz",
    bucket_name,
    f"{input_path}/grafana/grafana-12.0.1.linux-amd64.tar.gz",
)

print("Grafana binary uploaded to:")
print(grafana_s3_path)

***

# Step 2 - Run the job

In [ ]:
! pygmentize ./scripts/inference.py

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session

In [ ]:
sagemaker_session = Session()

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
from sagemaker.train.configs import InstanceGroup

instance_groups = [
    InstanceGroup(
        instance_group_name="head-instance-group",
        instance_type="ml.t3.large",
        instance_count=1,
    ),
    InstanceGroup(
        instance_group_name="worker-instance-group-1",
        instance_type="ml.m5.2xlarge",
        instance_count=2,
    ),
]

instance_groups

In [ ]:
image_uri = image_uris.retrieve(
    framework="pytorch",
    region=sagemaker_session.boto_session.region_name,
    version="2.8.0",
    instance_type=instance_groups[1].instance_type,
    image_scope="training",
)

image_uri

In [ ]:
from sagemaker.train.configs import (
    Compute,
    OutputDataConfig,
    RemoteDebugConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.train.model_trainer import ModelTrainer

args = [
    "--entrypoint",
    "inference.py",
    "--subset",
    "1000",
    # "--prometheus-path", # Enable these parameters in case of prometheus binary passed as InputData
    # "/opt/ml/input/data/prometheus/prometheus-3.13.1.linux-amd64.tar.gz", # Enable these parameters in case of prometheus binary passed as InputData
    # "--grafana-path", # Enable these parameters in case of grafana binary passed as InputData
    # "/opt/ml/input/data/grafana/grafana-12.0.1.linux-amd64.tar.gz", # Enable these parameters in case of grafana binary passed as InputData
]

# Define the script to be run
source_code = SourceCode(
    source_dir="./scripts",
    requirements="requirements.txt",
    command=f"python launcher.py {' '.join(args)}",
)

# Define the compute
compute_configs = Compute(
    instance_groups=instance_groups,
)

# define Training Job Name
job_name = "train-ray-data"

# define OutputDataConfig path
if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{job_name}"
else:
    output_path = f"s3://{bucket_name}/{job_name}"

# Define the ModelTrainer
model_trainer = ModelTrainer(
    training_image=image_uri,
    source_code=source_code,
    base_job_name=job_name,
    compute=compute_configs,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=18000),
    output_data_config=OutputDataConfig(
        s3_output_path=output_path, compression_type="NONE"
    ),
    environment={
        "head_instance_group": "head-instance-group",
        "head_num_cpus": "0",
        "head_num_gpus": "0",
        # "launch_prometheus": "false", # disable for local prometheus
        # "RAY_PROMETHEUS_HOST": "<PROMETHEUS_HOST>", # URL for remote prometheus server
        # "RAY_PROMETHEUS_NAME": "prometheus",
    },
    role=role,
).with_remote_debug_config(RemoteDebugConfig(enable_remote_debug=True))

In [ ]:
from sagemaker.train.configs import InputData, S3DataSource

## Uncomment this lines if you want to provide the prometheus binary

# prometheus_input = InputData(
#     channel_name="prometheus",
#     data_source=S3DataSource(
#         s3_data_type="S3Prefix",
#         s3_uri=prometheus_s3_path,
#         s3_data_distribution_type="FullyReplicated",
#         instance_group_names=[
#             "head-instance-group",
#         ],
#     ),  # S3 path where prometheus_s3_path binary is stored
# )

## Uncomment this lines if you want to provide the grafana binary

# grafana_input = InputData(
#     channel_name="grafana",
#     data_source=S3DataSource(
#         s3_data_type="S3Prefix",
#         s3_uri=grafana_s3_path,
#         s3_data_distribution_type="FullyReplicated",
#         instance_group_names=[
#             "head-instance-group",
#         ],
#     ),  # S3 path where grafana_s3_path binary is stored
# )

# Check input channels configured
data = [
    # prometheus_input,
    # grafana_input,
]
data

In [ ]:
# starting the train job with our uploaded datasets as input
model_trainer.train(input_data_config=data, wait=False)